# Study 927 — Dutch Auction — the teardown

145 issuer *modified Dutch auction* self-tenders on 109 distinct issuers, 2010-06-18 → 2025-11-21. Event set built by one EDGAR full-text-search query (`q="modified Dutch auction"`, `forms=SC TO-I`), clustered per registrant, event date = the earliest SC TO-I filing of each cluster (the commencement date, public that session). Abnormal return = issuer − SPY on daily **total-return** closes (`auto_adjust=True`). One execution lag: filing public at the close of *t*, tradable leg entered at the close of *t+1*.

Every real number is frozen from `docs/results.md` (Fingerprint `c39ea6c65ccb`); the live cells at the end are the **synthetic** control.

In [1]:
R = {'n_edgar': 180, 'n_events': 145, 'n_issuers': 109, 'n_tapes': 128, 'start': '2010-06-18', 'end': '2025-11-21', 'fp': 'c39ea6c65ccb', 'pre5': 1.44, 'pre5_med': 0.88, 'pre5_t': 3.2, 'pre5_hit': 58, 'ar0': 4.72, 'ar0_med': 3.26, 'ar0_t': 7.93, 'ar0_hit': 83, 'win': 0.12, 'win_med': 0.61, 'win_t': 0.17, 'win_hit': 55, 'm1': -0.25, 'm1_med': -1.08, 'm1_t': -0.29, 'm1_hit': 43, 'm3': 1.69, 'm3_med': -0.28, 'm3_t': 0.88, 'm3_hit': 50, 'm6': 2.76, 'm6_med': -2.36, 'm6_t': 0.71, 'm6_hit': 47, 'jk_ar0_lo': 7.81, 'jk_ar0_hi': 8.86, 'jk_win_lo': -0.04, 'jk_win_hi': 0.61, 'jk_m6_lo': -0.12, 'jk_m6_hi': 0.83, 'ci_ar0_lo': 3.55, 'ci_ar0_hi': 6.01, 'ci_win_lo': -1.45, 'ci_win_hi': 1.44, 'ci_m6_lo': -4.55, 'ci_m6_hi': 11.95, 'ci_win_neg': 41.4, 'ci_m6_neg': 29.2, 'pb_ar0_mean': 0.01, 'pb_ar0_sd': 0.29, 'pb_ar0_p': 0.0005, 'pb_win_p': 0.9105, 'pb_m6_p': 0.3025, 'cal_g_bps': -1.232, 'cal_g_sharpe': -0.169, 'cal_g_t': -0.63, 'cal_g_cum': -54.16, 'cal_n_bps': -2.213, 'cal_n_sharpe': -0.302, 'cal_n_t': -1.14, 'cal_n_cum': -69.24, 'cal_rt': 93.7, 'cal_overstate': 1.55, 'cal_drag': 0.926, 'cal_vol': 18.4, 'cal_live': 1847, 'cal_days': 4046, 'cal_names': 1.57, 'race_g': 0.168, 'race_n': 0.111, 'race_spy': 0.809, 'race_adv': -0.698, 'race_t': -2.0, 'race_spy_t': 3.6, 'race_n_vol': 21.43, 'race_spy_vol': 17.13, 'era_e_n': 63, 'era_e_ar0': 3.1, 'era_e_ar0_t': 5.78, 'era_e_win': 1.16, 'era_e_win_t': 1.53, 'era_e_m6': -1.15, 'era_e_m6_t': -0.36, 'era_l_n': 82, 'era_l_ar0': 5.97, 'era_l_ar0_t': 6.29, 'era_l_win': -0.69, 'era_l_win_t': -0.65, 'era_l_m6': 5.76, 'era_l_m6_t': 0.9, 'liq_n': 60, 'liq_ar0': 3.43, 'liq_ar0_t': 4.88, 'liq_ar0_hit': 75, 'liq_win': 0.14, 'liq_win_t': 0.13, 'liq_m6': 0.78, 'liq_m6_t': 0.24, 'liq_m6_med': -2.89, 'liq_cal_sharpe': -0.177, 'liq_cal_t': -0.59, 'cost0_win': 0.12, 'cost0_sharpe': -0.176, 'cost10_win': -0.28, 'cost10_sharpe': -0.302, 'cost25_win': -0.88, 'cost25_sharpe': -0.489, 'cost50_win': -1.88, 'cost50_sharpe': -0.792, 'borrow0': -0.295, 'borrow300': -0.369, 'shift_m5': -0.32, 'shift_m1': 0.64, 'shift_m1_t': 2.51, 'shift_p1': 0.43, 'shift_p1_t': 1.47, 'shift_p5': 0.0, 'shift_p5_t': 0.03, 'exp15_win': 0.64, 'exp15_m6': 1.49, 'exp25_win': -0.38, 'exp25_m6': 2.89, 'exp30_win': -0.16, 'exp30_m6': 1.96, 'syn_jump': 470, 'syn_rec': 468, 'syn_rec_t': 23.06, 'syn_drift': 300, 'syn_drift_exp': 607, 'syn_drift_rec': 661, 'syn_drift_t': 2.74, 'syn_null_ar0_max': 2.33, 'syn_null_ar0_fire': 2, 'syn_null_m6_max': 2.86, 'syn_null_m6_fire': 7, 'syn_null_n': 20}

## The window decomposition

Four separate objects, deliberately never blended: the pre-announcement run-up (SC TO-C / press-release leakage), the announcement-day repricing (news, not an edge), the tradable tender window, and the post-expiry drift.

> 💡 **In plain words.** We split the event into "what happened before anyone could act", "what happened the instant it became public", and "what happened afterwards, when you could actually buy".

In [2]:
rows = [('pre5  (-5,-1) run-up', R['pre5'], R['pre5_med'], R['pre5_t'], R['pre5_hit']),
        ('ar0   day 0 (news)  ', R['ar0'],  R['ar0_med'],  R['ar0_t'],  R['ar0_hit']),
        ('window t+1 -> expiry', R['win'],  R['win_med'],  R['win_t'],  R['win_hit']),
        ('m1    expiry +1m    ', R['m1'],   R['m1_med'],   R['m1_t'],   R['m1_hit']),
        ('m3    expiry +3m    ', R['m3'],   R['m3_med'],   R['m3_t'],   R['m3_hit']),
        ('m6    expiry +6m    ', R['m6'],   R['m6_med'],   R['m6_t'],   R['m6_hit'])]
print(f"n = {R['n_events']} events on {R['n_issuers']} issuers\n")
print(f"{'window':22s} {'mean':>8s} {'median':>8s} {'t':>7s} {'hit':>5s}")
for lab, mu, med, t, hit in rows:
    print(f'{lab:22s} {mu:+7.2f}% {med:+7.2f}% {t:+7.2f} {hit:4d}%')

n = 145 events on 109 issuers

window                     mean   median       t   hit
pre5  (-5,-1) run-up     +1.44%   +0.88%   +3.20   58%
ar0   day 0 (news)       +4.72%   +3.26%   +7.93   83%
window t+1 -> expiry     +0.12%   +0.61%   +0.17   55%
m1    expiry +1m         -0.25%   -1.08%   -0.29   43%
m3    expiry +3m         +1.69%   -0.28%   +0.88   50%
m6    expiry +6m         +2.76%   -2.36%   +0.71   47%


## Is day 0 robust? Jackknife, block bootstrap, placebo, date shift

Four checks, each attacking a different failure mode: one lucky event, a fat tail, a mis-specified null, and a mis-dated event.

In [3]:
print('jackknife (leave-one-out t)')
print(f"  ar0    t={R['ar0_t']:+.2f}  LOO [{R['jk_ar0_lo']:+.2f}, {R['jk_ar0_hi']:+.2f}]")
print(f"  window t={R['win_t']:+.2f}  LOO [{R['jk_win_lo']:+.2f}, {R['jk_win_hi']:+.2f}]")
print(f"  m6     t={R['m6_t']:+.2f}  LOO [{R['jk_m6_lo']:+.2f}, {R['jk_m6_hi']:+.2f}]")
print('\nblock bootstrap on the mean (5,000 draws, 5-event blocks)')
print(f"  ar0    {R['ar0']:+.2f}%  CI [{R['ci_ar0_lo']:+.2f}%, {R['ci_ar0_hi']:+.2f}%]")
print(f"  window {R['win']:+.2f}%  CI [{R['ci_win_lo']:+.2f}%, {R['ci_win_hi']:+.2f}%]  "
      f"({R['ci_win_neg']:.1f}% of draws negative)")
print(f"  m6     {R['m6']:+.2f}%  CI [{R['ci_m6_lo']:+.2f}%, {R['ci_m6_hi']:+.2f}%]  "
      f"({R['ci_m6_neg']:.1f}% negative)")
print('\nplacebo: same names, dates re-drawn at random on their own tapes (2,000 draws)')
print(f"  ar0    obs {R['ar0']:+.2f}%  placebo {R['pb_ar0_mean']:+.2f}% "
      f"(sd {R['pb_ar0_sd']:.2f}%)  p={R['pb_ar0_p']:.4f}")
print(f"  window p={R['pb_win_p']:.4f}   m6 p={R['pb_m6_p']:.4f}   <- both indistinguishable from chance")
print('\nevent-date shift (trading days)')
print(f"  -5: {R['shift_m5']:+.2f}%   -1: {R['shift_m1']:+.2f}% (t={R['shift_m1_t']:+.2f})   "
      f"0: {R['ar0']:+.2f}% (t={R['ar0_t']:+.2f})   +1: {R['shift_p1']:+.2f}% "
      f"(t={R['shift_p1_t']:+.2f})   +5: {R['shift_p5']:+.2f}%")

jackknife (leave-one-out t)
  ar0    t=+7.93  LOO [+7.81, +8.86]
  window t=+0.17  LOO [-0.04, +0.61]
  m6     t=+0.71  LOO [-0.12, +0.83]

block bootstrap on the mean (5,000 draws, 5-event blocks)
  ar0    +4.72%  CI [+3.55%, +6.01%]
  window +0.12%  CI [-1.45%, +1.44%]  (41.4% of draws negative)
  m6     +2.76%  CI [-4.55%, +11.95%]  (29.2% negative)

placebo: same names, dates re-drawn at random on their own tapes (2,000 draws)
  ar0    obs +4.72%  placebo +0.01% (sd 0.29%)  p=0.0005
  window p=0.9105   m6 p=0.3025   <- both indistinguishable from chance

event-date shift (trading days)
  -5: -0.32%   -1: +0.64% (t=+2.51)   0: +4.72% (t=+7.93)   +1: +0.43% (t=+1.47)   +5: +0.00%


## The overlapping-event problem, and the calendar-time fix

Events cluster in calendar time (2015 and 2021 were busy), so the cross-event one-sample *t* is not correctly sized at long horizons. The standard fix (Mitchell & Stafford 2000) is a calendar-time portfolio: equal-weight every name currently inside its window, take the daily series, and use a HAC *t*.

> 💡 **In plain words.** If ten of your events happen in the same quarter, they are really one bet on that quarter, not ten independent bets. The calendar-time portfolio counts it as one.

**How the costs are charged, and why it matters.** The sleeve is equal-weight across whichever names are live, so a slot sharing the book with two others is a third of NAV and can only cost a third of a round trip. Costs are therefore booked at each slot's **real portfolio weight**, on the day it opens and the day it closes: the 145 events come to **93.7** full-NAV round trips, a drag of 0.93 bps/day. Charging one full-NAV round trip per event — the obvious shortcut — would have inflated that by **1.55×** and handed the negative verdict to the cost model instead of the tape. It does not come to that: the sleeve is negative **-0.169 gross**, before a single basis point is charged.

In [4]:
print('long issuer / short SPY, entered t+1, held to the expiry proxy')
print(f"  gross: {R['cal_g_bps']:+.3f} bps/day  Sharpe {R['cal_g_sharpe']:+.3f}  "
      f"HAC t {R['cal_g_t']:+.2f}  cum {R['cal_g_cum']:+.2f}%")
print(f"  net  : {R['cal_n_bps']:+.3f} bps/day  Sharpe {R['cal_n_sharpe']:+.3f}  "
      f"HAC t {R['cal_n_t']:+.2f}  cum {R['cal_n_cum']:+.2f}%   (10 bps/leg, 30 bps/yr borrow)")
print(f"  live on {R['cal_live']:,} of {R['cal_days']:,} sessions, "
      f"{R['cal_names']:.2f} names on an average live day, vol {R['cal_vol']:.1f}%")
print('\nlong-only race, excess-of-cash (BIL) vs SPY')
print(f"  event basket gross : exSharpe {R['race_g']:+.3f}")
print(f"  event basket net   : exSharpe {R['race_n']:+.3f}  vol {R['race_n_vol']:.1f}%")
print(f"  SPY                : exSharpe {R['race_spy']:+.3f}  vol {R['race_spy_vol']:.1f}%  "
      f"HAC t {R['race_spy_t']:+.2f}")
print(f"  advantage {R['race_adv']:+.3f}   HAC t on the daily difference {R['race_t']:+.2f}")

long issuer / short SPY, entered t+1, held to the expiry proxy
  gross: -1.232 bps/day  Sharpe -0.169  HAC t -0.63  cum -54.16%
  net  : -2.213 bps/day  Sharpe -0.302  HAC t -1.14  cum -69.24%   (10 bps/leg, 30 bps/yr borrow)
  live on 1,847 of 4,046 sessions, 1.57 names on an average live day, vol 18.4%

long-only race, excess-of-cash (BIL) vs SPY
  event basket gross : exSharpe +0.168
  event basket net   : exSharpe +0.111  vol 21.4%
  SPY                : exSharpe +0.809  vol 17.1%  HAC t +3.60
  advantage -0.698   HAC t on the daily difference -2.00


## Era cut, liquidity cut, and the sweeps

The announcement effect must survive an era split and the liquid subset; the tradable legs must survive nothing, because there is nothing there. Every assumption that is not a tape input — cost, borrow, the expiry proxy — is swept.

In [5]:
print('era cut (split 2018-01-01)')
print(f"  2010-2017 (n={R['era_e_n']}): ar0 {R['era_e_ar0']:+.2f}% (t={R['era_e_ar0_t']:+.2f})  "
      f"window {R['era_e_win']:+.2f}% (t={R['era_e_win_t']:+.2f})  m6 {R['era_e_m6']:+.2f}% (t={R['era_e_m6_t']:+.2f})")
print(f"  2018-2025 (n={R['era_l_n']}): ar0 {R['era_l_ar0']:+.2f}% (t={R['era_l_ar0_t']:+.2f})  "
      f"window {R['era_l_win']:+.2f}% (t={R['era_l_win_t']:+.2f})  m6 {R['era_l_m6']:+.2f}% (t={R['era_l_m6_t']:+.2f})")
print(f"\nliquidity subset (median 60d dollar volume >= $10m, n={R['liq_n']})")
print(f"  ar0 {R['liq_ar0']:+.2f}% (t={R['liq_ar0_t']:+.2f}, hit {R['liq_ar0_hit']}%)  "
      f"window {R['liq_win']:+.2f}% (t={R['liq_win_t']:+.2f})  "
      f"m6 {R['liq_m6']:+.2f}% (median {R['liq_m6_med']:+.2f}%, t={R['liq_m6_t']:+.2f})")
print(f"  calendar-time net Sharpe {R['liq_cal_sharpe']:+.3f} (HAC t {R['liq_cal_t']:+.2f})")
print('\ncost sweep (one-way x NAV, both legs, in and out)')
for c, w, s in [(0, R['cost0_win'], R['cost0_sharpe']), (10, R['cost10_win'], R['cost10_sharpe']),
                (25, R['cost25_win'], R['cost25_sharpe']), (50, R['cost50_win'], R['cost50_sharpe'])]:
    print(f'  {c:2d} bps: net window {w:+.2f}%   calendar-time Sharpe {s:+.3f}')
print(f"\nborrow sweep (short-SPY leg, an ASSUMPTION): Sharpe {R['borrow0']:+.3f} at 0 bps/yr "
      f"-> {R['borrow300']:+.3f} at 300 bps/yr")
print('\nexpiry PROXY sweep (Rule 14e-1 minimum = 20 sessions)')
print(f"  +15td: window {R['exp15_win']:+.2f}%  m6 {R['exp15_m6']:+.2f}%")
print(f"  +20td: window {R['win']:+.2f}%  m6 {R['m6']:+.2f}%   <- headline")
print(f"  +25td: window {R['exp25_win']:+.2f}%  m6 {R['exp25_m6']:+.2f}%")
print(f"  +30td: window {R['exp30_win']:+.2f}%  m6 {R['exp30_m6']:+.2f}%")

era cut (split 2018-01-01)
  2010-2017 (n=63): ar0 +3.10% (t=+5.78)  window +1.16% (t=+1.53)  m6 -1.15% (t=-0.36)
  2018-2025 (n=82): ar0 +5.97% (t=+6.29)  window -0.69% (t=-0.65)  m6 +5.76% (t=+0.90)

liquidity subset (median 60d dollar volume >= $10m, n=60)
  ar0 +3.43% (t=+4.88, hit 75%)  window +0.14% (t=+0.13)  m6 +0.78% (median -2.89%, t=+0.24)
  calendar-time net Sharpe -0.177 (HAC t -0.59)

cost sweep (one-way x NAV, both legs, in and out)
   0 bps: net window +0.12%   calendar-time Sharpe -0.176
  10 bps: net window -0.28%   calendar-time Sharpe -0.302
  25 bps: net window -0.88%   calendar-time Sharpe -0.489
  50 bps: net window -1.88%   calendar-time Sharpe -0.792

borrow sweep (short-SPY leg, an ASSUMPTION): Sharpe -0.295 at 0 bps/yr -> -0.369 at 300 bps/yr

expiry PROXY sweep (Rule 14e-1 minimum = 20 sessions)
  +15td: window +0.64%  m6 +1.49%
  +20td: window +0.12%  m6 +2.76%   <- headline
  +25td: window -0.38%  m6 +2.89%
  +30td: window -0.16%  m6 +1.96%


## Live synthetic control — recover the plant, stay quiet on the null

**Synthetic, not the real tape.** 60 issuers on one shared market factor. The planted world carries a day-0 jump and a six-month drift; the null carries neither. The harness must recover the first and find nothing in the second.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from dutch_auction import data, strategy as st

px, ev, truth = data.synthetic_panel(n_events=60, n_days=900,
                                     signal_strength=1.0, seed=927)
d = st.synthetic_detect(px, ev, market='MKT')
print('SYNTHETIC planted world (not the real tape)')
print('  day-0 : planted %+.0f bps -> recovered %+.0f bps (t=%+.2f)'
      % (truth['planted_jump']*1e4, d['ar0_bps'], d['t_ar0']))
print('  6m    : planted %+.0f bps of LOG drift = %+.0f bps of expected SIMPLE'
      % (truth['planted_drift_6m_log']*1e4, truth['expected_simple_drift_6m']*1e4))
print('          buy-and-hold abnormal return (Jensen) -> recovered %+.0f bps (t=%+.2f)'
      % (d['m6_bps'], d['t_m6']))

ts = []
for s in range(12):
    npx, nev, _ = data.synthetic_panel(n_events=60, n_days=900,
                                       signal_strength=0.0, seed=927+s)
    ts.append(st.synthetic_detect(npx, nev, market='MKT')['t_ar0'])
ts = np.array(ts)
print('SYNTHETIC null x12: mean t(day-0) %+.2f, max |t| %.2f, |t|>=2 on %d/12'
      % (ts.mean(), np.abs(ts).max(), (np.abs(ts) >= 2).sum()))

SYNTHETIC planted world (not the real tape)
  day-0 : planted +470 bps -> recovered +470 bps (t=+17.28)
  6m    : planted +300 bps of LOG drift = +607 bps of expected SIMPLE
          buy-and-hold abnormal return (Jensen) -> recovered +1195 bps (t=+3.28)


SYNTHETIC null x12: mean t(day-0) -0.11, max |t| 1.25, |t|>=2 on 0/12


## A finding hiding in the null

On the headline run the null was pushed to 20 seeds, and the day-0 statistic behaved (|*t*| max 2.33, ≥2 on 2/20) while the **six-month** statistic did not: |*t*| max 2.86, ≥2 on 7/20. That is not a bug — it is Barber-Lyon and Kothari-Warner reproduced in miniature. A six-month *simple* buy-and-hold abnormal return is right-skewed and positively centred even under a null (the same Jensen term that turns a planted 300 bps of log drift into 607 bps of expected simple return), and the shared market factor makes those returns cross-sectionally correlated on top. So the naive cross-event *t* **over-rejects** at long horizons. The correctly sized tests for the drift are therefore the placebo (*p* = 0.3025) and the calendar-time HAC *t* (-1.14) — both empty — and the real-tape six-month *t* of +0.71 is, if anything, flattered.

## Verdict

- **Signal — Mixed.** The **announcement repricing is Real**: +4.72% abnormal on the SC TO-I session, one-sample *t* = +7.93, jackknife LOO [+7.81, +8.86], bootstrap CI [+3.55%, +6.01%], placebo *p* = 0.0005, date-locked to a single session, present in both eras (+3.10% / +5.97%) and in the liquid subset (+3.43%, *t* = +4.88). The **bottom-marking claim is None**: tender window +0.12% (*t* = +0.17, placebo *p* = 0.91), six-month drift +2.76% with a -2.36% median (*t* = +0.71, CI [-4.55%, +11.95%]). Named biases: the CIK→ticker map is SEC's **current** register (survivorship); the screens need 148 sessions of tape AFTER the event, so an issuer taken out within ~7 months of its own tender is dropped — exactly the outcome that would print a big positive six-month drift, which makes the flat drift above an **upper bound**; and the sample is the subset of self-tenders whose filings say "modified Dutch auction" (visibility).
- **Tradability — Mirage.** The paying session is the one you cannot be positioned for. The calendar-time long/short compounds to -69.2% net (-0.302 Sharpe, HAC *t* -1.14); the long-only basket trails SPY by -0.698 Sharpe with a HAC *t* of -2.00 on the daily difference; cost and borrow sweeps only make it worse; no expiry assumption rescues it. The premium is paid to the holders who tender, not to the tape.